# US 수출 데이터 SARIMA 예측 및 DB 저장

이 노트북은 US 수출 데이터(expDlr)를 HS 코드별로 SARIMA 예측하고 MySQL DB에 저장합니다.

## 주요 기능
1. HS 코드별 월별 expDlr SARIMA 예측
2. 월별 → 분기별 자동 집계
3. 예측 파라미터 JSON 저장
4. created_at 기준 버전 관리
5. tqdm 진행바로 실시간 모니터링

## 1. 라이브러리 및 함수 정의

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from datetime import datetime
from typing import Optional, Dict, Tuple
import json

from statsmodels.tsa.statespace.sarimax import SARIMAX
from pandas.tseries.offsets import MonthEnd
from itertools import product
from sqlalchemy import create_engine, text
import pymysql
from tqdm import tqdm

print("라이브러리 로드 완료")

라이브러리 로드 완료


### 1.1 유틸리티 함수

In [2]:
def to_month_end(s):
    """날짜를 월말로 변환"""
    ts = pd.to_datetime(s)
    if isinstance(ts, pd.Timestamp):
        return ts + MonthEnd(0)
    return ts + MonthEnd(0)


def ensure_sorted_unique_dates(df: pd.DataFrame, date_col: str = "time") -> pd.DataFrame:
    """날짜 정렬 및 중복 제거"""
    d = df.copy()
    d[date_col] = pd.to_datetime(d[date_col])
    d[date_col] = d[date_col] + MonthEnd(0)
    return d.sort_values(date_col).drop_duplicates([date_col]).reset_index(drop=True)


print("유틸리티 함수 정의 완료")

유틸리티 함수 정의 완료


### 1.2 SARIMA 파라미터 탐색

In [3]:
def find_best_sarima_params(
    y_train: pd.Series,
    seasonal_period: int = 12,
    p_values=(0, 1, 2),
    d_values=(0, 1),
    q_values=(0, 1, 2),
    P_values=(0, 1),
    D_values=(0, 1),
    Q_values=(0, 1),
    ic: str = "aic",
    max_order_sum: int = 8,
) -> Tuple[tuple, tuple]:
    """최적 SARIMA 파라미터 탐색"""
    best_ic = np.inf
    best_order = (1, 1, 1)
    best_sorder = (1, 1, 0, seasonal_period)

    for p, d, q in product(p_values, d_values, q_values):
        for P, D, Q in product(P_values, D_values, Q_values):
            if (p + q + P + Q) > max_order_sum:
                continue
            order = (p, d, q)
            sorder = (P, D, Q, seasonal_period)
            try:
                m = SARIMAX(
                    y_train.astype(float),
                    order=order,
                    seasonal_order=sorder,
                    enforce_stationarity=False,
                    enforce_invertibility=False
                )
                fit = m.fit(disp=False)
                val = fit.aic if ic.lower() == "aic" else fit.bic
                if np.isfinite(val) and val < best_ic:
                    best_ic, best_order, best_sorder = val, order, sorder
            except Exception:
                continue

    return best_order, best_sorder


print("파라미터 탐색 함수 정의 완료")

파라미터 탐색 함수 정의 완료


### 1.3 월별 SARIMA 예측

In [4]:
def run_sarima_forecast_monthly(
    df: pd.DataFrame,
    hs_code: str,
    forecast_months: int = 18,
    target_col: str = "expDlr",
    ic: str = "aic",
    min_obs: int = 24,
    date_col: str = "time"
):
    """
    월별 SARIMA 예측 수행

    Parameters:
    -----------
    df : DataFrame with columns ['time', 'expDlr']
    hs_code : HS 코드
    forecast_months : 예측할 개월 수
    target_col : 예측 대상 컬럼
    ic : 정보기준 ('aic' or 'bic')
    min_obs : 최소 관측치 수
    date_col : 날짜 컬럼명

    Returns:
    --------
    result_df : 예측 결과 포함 DataFrame
    metadata : 모델 정보 딕셔너리
    """
    results = {
        "hs_code": hs_code,
        "model": "SARIMA",
        "frequency": "monthly",
        "order": None,
        "seasonal_order": None,
        "aic": None,
        "bic": None,
        "error": None
    }

    try:
        # 날짜 컬럼 자동 감지
        if date_col not in df.columns:
            if 'date' in df.columns:
                date_col = 'date'
            elif 'time' in df.columns:
                date_col = 'time'
            else:
                raise ValueError("날짜 컬럼을 찾을 수 없습니다.")

        # 데이터 정렬 및 정제
        d = ensure_sorted_unique_dates(df[[date_col, target_col]], date_col)
        d = d.rename(columns={date_col: "date_month_end"})

        # 시계열 생성
        y = pd.Series(d[target_col].values, index=d["date_month_end"]).dropna()

        if len(y) < min_obs:
            results["error"] = f"관측치 부족: {len(y)} < {min_obs}"
            d["expDlr_forecast"] = d[target_col]
            return d, results

        # 최적 파라미터 탐색
        order, sorder = find_best_sarima_params(
            y, seasonal_period=12, ic=ic
        )

        # 모델 적합
        model = SARIMAX(
            y,
            order=order,
            seasonal_order=sorder,
            enforce_stationarity=False,
            enforce_invertibility=False
        )
        fit = model.fit(disp=False)

        # 예측
        forecast = fit.forecast(steps=forecast_months)

        # 미래 날짜 생성
        last_date = y.index.max()
        future_dates = pd.date_range(
            last_date + MonthEnd(1),
            periods=forecast_months,
            freq="M"
        )

        # 결과 DataFrame 생성
        result_df = d.copy()
        result_df["expDlr_forecast"] = result_df[target_col]

        # 미래 예측값 추가
        for i, future_date in enumerate(future_dates):
            if future_date not in result_df["date_month_end"].values:
                new_row = pd.DataFrame({
                    "date_month_end": [future_date],
                    target_col: [np.nan],
                    "expDlr_forecast": [float(forecast.iloc[i])]
                })
                result_df = pd.concat([result_df, new_row], ignore_index=True)
            else:
                result_df.loc[
                    result_df["date_month_end"] == future_date,
                    "expDlr_forecast"
                ] = float(forecast.iloc[i])

        # 메타데이터 저장
        results["order"] = order
        results["seasonal_order"] = sorder
        results["aic"] = float(fit.aic)
        results["bic"] = float(fit.bic)

        # 정렬
        result_df = result_df.sort_values("date_month_end").reset_index(drop=True)

    except Exception as e:
        results["error"] = str(e)
        result_df = d.copy() if 'd' in locals() else df.copy()
        if "expDlr_forecast" not in result_df.columns:
            result_df["expDlr_forecast"] = result_df.get(target_col)

    return result_df, results

### 1.4 분기별 집계

In [5]:
def aggregate_to_quarter(monthly_df: pd.DataFrame) -> pd.DataFrame:
    """
    월별 데이터를 분기별로 집계

    Parameters:
    -----------
    monthly_df : 월별 데이터 (date_month_end, expDlr, expDlr_forecast 컬럼 필요)

    Returns:
    --------
    quarterly_df : 분기별 집계 데이터
    """
    df = monthly_df.copy()
    df["quarter"] = df["date_month_end"].dt.to_period("Q")

    # 분기별 합계
    quarterly = df.groupby("quarter").agg({
        "expDlr": "sum",
        "expDlr_forecast": "sum"
    }).reset_index()

    # 분기 말일로 변환
    quarterly["date_quarter_end"] = quarterly["quarter"].dt.to_timestamp(how="end")
    quarterly["date_quarter_end"] = quarterly["date_quarter_end"] + MonthEnd(0)

    quarterly = quarterly.drop("quarter", axis=1)

    return quarterly


print("분기별 집계 함수 정의 완료")

분기별 집계 함수 정의 완료


### 1.5 DB 저장용 데이터 준비

In [6]:
def prepare_db_table_monthly(
    df: pd.DataFrame,
    hs_code: str,
    metadata: Dict,
    created_at: datetime
) -> pd.DataFrame:
    """
    월별 예측 결과를 DB 저장용 형태로 변환

    Columns: hs_code, date_month_end, expDlr, expDlr_forecast,
             is_forecast, params, created_at
    """
    db_df = df.copy()
    db_df["hs_code"] = hs_code
    db_df["is_forecast"] = db_df["expDlr"].isna().astype(int)

    # 파라미터를 JSON 문자열로 저장
    params_dict = {
        "model": metadata.get("model"),
        "order": metadata.get("order"),
        "seasonal_order": metadata.get("seasonal_order"),
        "aic": metadata.get("aic"),
        "bic": metadata.get("bic")
    }
    db_df["params"] = json.dumps(params_dict, ensure_ascii=False)
    db_df["created_at"] = created_at

    # 컬럼 순서 정리
    db_df = db_df[[
        "hs_code", "date_month_end", "expDlr", "expDlr_forecast",
        "is_forecast", "params", "created_at"
    ]]

    return db_df


def prepare_db_table_quarter(
    df: pd.DataFrame,
    hs_code: str,
    metadata: Dict,
    created_at: datetime
) -> pd.DataFrame:
    """
    분기별 예측 결과를 DB 저장용 형태로 변환

    Columns: hs_code, date_quarter_end, expDlr, expDlr_forecast,
             is_forecast, params, created_at
    """
    db_df = df.copy()
    db_df["hs_code"] = hs_code
    db_df["is_forecast"] = db_df["expDlr"].isna().astype(int)

    # 파라미터를 JSON 문자열로 저장
    params_dict = {
        "model": metadata.get("model"),
        "frequency": "quarterly",
        "order": metadata.get("order"),
        "seasonal_order": metadata.get("seasonal_order"),
        "aic": metadata.get("aic"),
        "bic": metadata.get("bic")
    }
    db_df["params"] = json.dumps(params_dict, ensure_ascii=False)
    db_df["created_at"] = created_at

    # 컬럼 순서 정리
    db_df = db_df[[
        "hs_code", "date_quarter_end", "expDlr", "expDlr_forecast",
        "is_forecast", "params", "created_at"
    ]]

    return db_df


print("DB 준비 함수 정의 완료")

DB 준비 함수 정의 완료


### 1.6 DB 저장 함수

In [7]:
def save_to_mysql(
    df: pd.DataFrame,
    table_name: str,
    db_info: Dict,
    if_exists: str = "append"
):
    """
    DataFrame을 MySQL DB에 저장

    Parameters:
    -----------
    df : 저장할 DataFrame
    table_name : 테이블 이름
    db_info : DB 연결 정보 {'host', 'user', 'password', 'database', 'port'}
    if_exists : 'append' or 'replace'
    """
    # SQLAlchemy 엔진 생성
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
        f"{db_info['host']}:{db_info.get('port', 3306)}/{db_info['database']}",
        echo=False
    )

    try:
        # DataFrame을 MySQL에 저장
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists=if_exists,
            index=False,
            chunksize=1000
        )
        print(f"저장 완료: {table_name} ({len(df):,} rows)")
    except Exception as e:
        print(f"DB 저장 실패: {str(e)}")
        raise
    finally:
        engine.dispose()


print("DB 저장 함수 정의 완료")

DB 저장 함수 정의 완료


### 1.7 전체 처리 메인 함수

In [8]:
# # ════════════════════════════════════════════════════════════════════════════
# # 래퍼: run_sarima_forecast_monthly 내부가 'expDlr' 하드코딩인 경우 대응
# # ════════════════════════════════════════════════════════════════════════════
# def _run_sarima_compat(df, hs_code, forecast_months, target_col, ic, min_obs):
#     """
#     run_sarima_forecast_monthly 가 내부에서 'expDlr' 를 하드코딩하는 경우,
#     호출 전 컬럼명을 'expDlr' 로 임시 변환하고 반환 후 되돌립니다.
#     """
#     df_tmp = df.copy()
#     if target_col != "expDlr" and target_col in df_tmp.columns:
#         df_tmp = df_tmp.rename(columns={target_col: "expDlr"})
#
#     monthly_df, metadata = run_sarima_forecast_monthly(
#         df=df_tmp,
#         hs_code=hs_code,
#         forecast_months=forecast_months,
#         target_col="expDlr",
#         ic=ic,
#         min_obs=min_obs,
#     )
#
#     if monthly_df is not None and "expDlr" in monthly_df.columns:
#         monthly_df = monthly_df.rename(columns={"expDlr": "exp_dlr"})
#
#     return monthly_df, metadata
#
#
# def forecast_and_save_all_hs_codes(
#     trade_data: pd.DataFrame,
#     db_info: Dict,
#     forecast_months: int = 12,
#     min_obs: int = 24,
#     ic: str = "aic",
#     date_col: str = "date"
# ):
#     """
#     모든 HS 코드에 대해 예측 수행 후 결과 DataFrame 반환
#     (DB 저장은 다음 단계에서 수행)
#
#     Parameters:
#     -----------
#     trade_data      : US 무역 데이터 (columns: hs_code, date, exp_dlr)
#     db_info         : DB 연결 정보 (이 단계 미사용, 시그니처 유지용)
#     forecast_months : 예측 개월 수
#     min_obs         : 최소 관측치 수
#     ic              : 정보 기준
#     date_col        : 날짜 컬럼명 (기본: 'date', 'time'일 수도 있음)
#
#     Returns:
#     --------
#     created_at  : datetime
#     all_monthly : pd.DataFrame  월별 실적+예측
#     all_quarter : pd.DataFrame  분기별 실적+예측
#     """
#     import datetime as dt
#     created_at = dt.datetime.now()
#
#     # date_col이 존재하는지 확인하고 자동 조정
#     if date_col not in trade_data.columns:
#         if 'date' in trade_data.columns:
#             date_col = 'date'
#         elif 'time' in trade_data.columns:
#             date_col = 'time'
#         else:
#             raise ValueError("날짜 컬럼을 찾을 수 없습니다. 'time' 또는 'date' 컬럼이 필요합니다.")
#
#     # HS 코드 목록
#     hs_codes = trade_data["hs_code"].unique()
#     total_codes = len(hs_codes)
#
#     print(f"총 {total_codes:,}개 HS 코드 처리 시작...")
#     print(f"예측 개월 수: {forecast_months}")
#     print(f"생성 시각: {created_at}")
#     print("=" * 80)
#
#     monthly_results = []
#     quarter_results = []
#     failed_codes = []
#
#     pbar = tqdm(hs_codes, desc="예측 진행", unit="HS코드")
#
#     for hs_code in pbar:
#         try:
#             pbar.set_postfix({"현재": str(hs_code)[:10], "성공": len(monthly_results), "실패": len(failed_codes)})
#
#             # HS 코드별 데이터 추출
#             hs_data = trade_data[trade_data["hs_code"] == hs_code].copy()
#
#             # 날짜 컬럼명을 'time'으로 통일
#             if date_col != 'time':
#                 hs_data = hs_data.rename(columns={date_col: 'time'})
#
#             # exp_dlr → log1p 변환 (분산 안정화)
#             hs_data["exp_dlr"] = np.log1p(hs_data["exp_dlr"].clip(lower=0))
#
#             # 월별 예측 — 래퍼를 통해 내부 하드코딩 'expDlr' 문제 우회
#             monthly_df, metadata = _run_sarima_compat(
#                 df=hs_data,
#                 hs_code=hs_code,
#                 forecast_months=forecast_months,
#                 target_col="exp_dlr",
#                 ic=ic,
#                 min_obs=min_obs,
#             )
#
#             if metadata.get("error"):
#                 failed_codes.append({"hs_code": hs_code, "error": metadata["error"]})
#                 continue
#
#             # log1p 역변환 (expm1) — 실적·예측 컬럼 모두 적용
#             for col in ["exp_dlr", "forecast", "lower_95", "upper_95"]:
#                 if col in monthly_df.columns:
#                     monthly_df[col] = np.expm1(monthly_df[col]).clip(lower=0)
#
#             # 분기별 집계
#             quarter_df = aggregate_to_quarter(monthly_df)
#
#             # DB 저장용 형태로 변환
#             db_monthly = prepare_db_table_monthly(
#                 monthly_df, hs_code, metadata, created_at
#             )
#             db_quarter = prepare_db_table_quarter(
#                 quarter_df, hs_code, metadata, created_at
#             )
#
#             monthly_results.append(db_monthly)
#             quarter_results.append(db_quarter)
#
#         except Exception as e:
#             failed_codes.append({"hs_code": hs_code, "error": str(e)})
#             continue
#
#     pbar.close()
#     print("=" * 80)
#
#     # 결과 요약
#     print(f"\n처리 완료:")
#     print(f"  성공: {len(monthly_results):,}개")
#     print(f"  실패: {len(failed_codes):,}개")
#
#     if failed_codes:
#         print(f"\n실패한 HS 코드 샘플 (최대 10개):")
#         for item in failed_codes[:10]:
#             print(f"  - {item['hs_code']}: {item['error']}")
#
#     # 결과 통합 후 반환 (DB 저장은 다음 단계)
#     all_monthly = pd.concat(monthly_results, ignore_index=True) if monthly_results else pd.DataFrame()
#     all_quarter = pd.concat(quarter_results, ignore_index=True) if quarter_results else pd.DataFrame()
#
#     print(f"\n월별 데이터: {len(all_monthly):,} rows")
#     print(f"분기별 데이터: {len(all_quarter):,} rows")
#     print("\n모든 처리 완료!")
#
#     return created_at, all_monthly, all_quarter
#
#
# print("메인 함수 정의 완료")
#
#
# # -*- coding: utf-8 -*-
# """
# debug_hs_854232.py
# -------------------
# hs_code=854232 단일 코드로 파이프라인 전 단계를 단계별로 확인합니다.
# 각 셀을 순서대로 실행하세요.
# """
#
# # ════════════════════════════════════════════════════════════════════════════
# # [셀 1] 데이터 추출 및 컬럼 확인
# # ════════════════════════════════════════════════════════════════════════════
# TARGET_HS = "854232"
#
# hs_data = trade_df[trade_df["hs_code"] == TARGET_HS].copy()
#
# print("=" * 60)
# print(f"HS 코드: {TARGET_HS}")
# print(f"행 수  : {len(hs_data)}")
# print(f"컬럼   : {list(hs_data.columns)}")
# print(f"\n--- head ---")
# print(hs_data.head())
# print(f"\n--- dtypes ---")
# print(hs_data.dtypes)
# print(f"\n--- exp_dlr 기술통계 ---")
# print(hs_data["exp_dlr"].describe())
#
#
# # ════════════════════════════════════════════════════════════════════════════
# # [셀 2] run_sarima_forecast_monthly 가 실제로 기대하는 컬럼 확인
# #         → 함수 소스코드에서 'expDlr' / 'exp_dlr' 검색
# # ════════════════════════════════════════════════════════════════════════════
# import inspect
#
# src = inspect.getsource(run_sarima_forecast_monthly)
# print("=== run_sarima_forecast_monthly 소스 내 컬럼명 검색 ===")
# for keyword in ["expDlr", "exp_dlr", "target_col", "time", "date"]:
#     count = src.count(keyword)
#     print(f"  '{keyword}' 출현 횟수: {count}")
#
# print("\n--- 함수 소스 전체 ---")
# print(src)
#
#
# # ════════════════════════════════════════════════════════════════════════════
# # [셀 3] 날짜 컬럼 rename 후 상태 확인
# # ════════════════════════════════════════════════════════════════════════════
# date_col = "date"
#
# hs_renamed = hs_data.copy()
# if date_col != "time":
#     hs_renamed = hs_renamed.rename(columns={date_col: "time"})
#
# print("rename 후 컬럼:", list(hs_renamed.columns))
# print(hs_renamed.head(3))
#
#
# # ════════════════════════════════════════════════════════════════════════════
# # [셀 4] log1p 변환 후 상태 확인
# # ════════════════════════════════════════════════════════════════════════════
# import numpy as np
#
# hs_log = hs_renamed.copy()
# hs_log["exp_dlr"] = np.log1p(hs_log["exp_dlr"].clip(lower=0))
#
# print("log1p 변환 후 exp_dlr 샘플:")
# print(hs_log["exp_dlr"].head(5).values)
# print(f"NaN 수: {hs_log['exp_dlr'].isna().sum()}")
#
#
# # ════════════════════════════════════════════════════════════════════════════
# # [셀 5] run_sarima_forecast_monthly 직접 호출 — 에러 재현
# # ════════════════════════════════════════════════════════════════════════════
# try:
#     monthly_df, metadata = run_sarima_forecast_monthly(
#         df=hs_log,
#         hs_code=TARGET_HS,
#         forecast_months=12,
#         target_col="exp_dlr",
#         ic="aic",
#         min_obs=24,
#     )
#     print("호출 성공")
#     print("metadata:", metadata)
#     print(monthly_df.head())
# except Exception as e:
#     print(f"에러 발생: {type(e).__name__}: {e}")
#     print("\n→ 함수 내부에서 'expDlr' 를 하드코딩하고 있을 가능성이 높습니다.")
#     print("  [셀 2] 출력의 소스코드에서 'expDlr' 위치를 확인하세요.")
#
#
# # ════════════════════════════════════════════════════════════════════════════
# # [셀 6] 근본 원인 수정 — run_sarima_forecast_monthly 래퍼로 우회
# #         함수 내부를 고칠 수 없을 때 사용
# # ════════════════════════════════════════════════════════════════════════════
# def run_sarima_forecast_monthly_fixed(df, hs_code, forecast_months,
#                                       target_col, ic, min_obs):
#     """
#     run_sarima_forecast_monthly 가 내부적으로 'expDlr' 를 하드코딩한 경우
#     호출 전에 컬럼명을 'expDlr' 로 임시 변환해서 넘깁니다.
#     반환된 DataFrame 은 다시 'exp_dlr' 로 되돌립니다.
#     """
#     df_tmp = df.copy()
#
#     # target_col(exp_dlr) → expDlr 로 임시 변환 (함수 내부 하드코딩 대응)
#     if target_col != "expDlr" and target_col in df_tmp.columns:
#         df_tmp = df_tmp.rename(columns={target_col: "expDlr"})
#
#     monthly_df, metadata = run_sarima_forecast_monthly(
#         df=df_tmp,
#         hs_code=hs_code,
#         forecast_months=forecast_months,
#         target_col="expDlr",   # 함수가 기대하는 이름 그대로
#         ic=ic,
#         min_obs=min_obs,
#     )
#
#     # 반환된 DataFrame 의 컬럼명 되돌리기
#     if monthly_df is not None and "expDlr" in monthly_df.columns:
#         monthly_df = monthly_df.rename(columns={"expDlr": "exp_dlr"})
#
#     return monthly_df, metadata
#
#
# # 래퍼로 재시도
# print("\n=== 래퍼 함수로 재시도 ===")
# try:
#     monthly_df, metadata = run_sarima_forecast_monthly_fixed(
#         df=hs_log,
#         hs_code=TARGET_HS,
#         forecast_months=12,
#         target_col="exp_dlr",
#         ic="aic",
#         min_obs=24,
#     )
#     print("성공!")
#     print("metadata:", metadata)
#     print(monthly_df.head())
#     print(f"\n컬럼: {list(monthly_df.columns)}")
# except Exception as e:
#     print(f"에러: {type(e).__name__}: {e}")

In [9]:
def forecast_and_save_all_hs_codes(
    trade_data: pd.DataFrame,
    db_info: Dict,
    forecast_months: int = 12,
    min_obs: int = 24,
    ic: str = "aic",
    date_col: str = "date"
):
    """
    모든 HS 코드에 대해 예측 수행 후 결과 DataFrame 반환
    (DB 저장은 다음 단계에서 수행)

    run_sarima_forecast_monthly 반환 컬럼 기준:
        date_month_end / exp_dlr / expDlr_forecast

    Returns:
    --------
    created_at  : datetime
    all_monthly : pd.DataFrame  월별 실적+예측
    all_quarter : pd.DataFrame  분기별 실적+예측
    """
    import datetime as dt
    created_at = dt.datetime.now()

    # date_col 자동 감지
    if date_col not in trade_data.columns:
        if 'date' in trade_data.columns:
            date_col = 'date'
        elif 'time' in trade_data.columns:
            date_col = 'time'
        else:
            raise ValueError("날짜 컬럼을 찾을 수 없습니다. 'time' 또는 'date' 컬럼이 필요합니다.")

    hs_codes    = trade_data["hs_code"].unique()
    total_codes = len(hs_codes)

    print(f"총 {total_codes:,}개 HS 코드 처리 시작...")
    print(f"예측 개월 수: {forecast_months}")
    print(f"생성 시각: {created_at}")
    print("=" * 80)

    monthly_results = []
    quarter_results = []
    failed_codes    = []

    pbar = tqdm(hs_codes, desc="예측 진행", unit="HS코드")

    for hs_code in pbar:
        try:
            pbar.set_postfix({
                "현재": str(hs_code)[:10],
                "성공": len(monthly_results),
                "실패": len(failed_codes)
            })

            # ── 데이터 추출 ───────────────────────────────────────────────
            hs_data = trade_data[trade_data["hs_code"] == hs_code].copy()

            # date_col → 'time' 으로 rename (함수가 date_col='time' 기본값)
            if date_col != 'time':
                hs_data = hs_data.rename(columns={date_col: 'time'})

            # exp_dlr → log1p 변환 후 expDlr 로 rename (함수 내부 컬럼명 맞춤)
            hs_data["exp_dlr"] = np.log1p(hs_data["exp_dlr"].clip(lower=0))
            hs_data = hs_data.rename(columns={"exp_dlr": "expDlr"})

            # ── 예측 ─────────────────────────────────────────────────────
            result_df, metadata = run_sarima_forecast_monthly(
                df=hs_data,
                hs_code=hs_code,
                forecast_months=forecast_months,
                target_col="expDlr",
                ic=ic,
                min_obs=min_obs,
            )

            if metadata.get("error"):
                failed_codes.append({"hs_code": hs_code, "error": metadata["error"]})
                continue

            # ── 반환 컬럼 정리 ────────────────────────────────────────────
            # 실제 반환: date_month_end / expDlr / expDlr_forecast
            # → 표준 컬럼명으로 통일
            df_m = result_df.rename(columns={
                "date_month_end":  "date",
                "expDlr":          "exp_dlr",
                "expDlr_forecast": "forecast",
            }).copy()

            # expm1 역변환 (달러 원본 복원)
            df_m["exp_dlr"] = np.expm1(df_m["exp_dlr"]).clip(lower=0)
            df_m["forecast"] = np.expm1(df_m["forecast"]).clip(lower=0)

            # is_forecast 구분 (실적 없는 행 = 예측)
            df_m["is_forecast"] = df_m["exp_dlr"].isna().astype(int)

            # 부가 컬럼
            df_m["hs_code"]     = hs_code
            df_m["year"]        = df_m["date"].dt.strftime("%Y")
            df_m["month"]       = df_m["date"].dt.strftime("%m")
            df_m["model_order"] = str(metadata.get("order")) + str(metadata.get("seasonal_order"))
            df_m["aic"]         = metadata.get("aic")
            df_m["created_at"]  = created_at

            # 컬럼 순서 정리
            df_m = df_m[[
                "hs_code", "date", "year", "month",
                "exp_dlr", "forecast", "is_forecast",
                "model_order", "aic", "created_at"
            ]]

            # ── 분기별 집계 ───────────────────────────────────────────────
            df_q = df_m.copy()
            df_q["quarter_start"] = df_q["date"].dt.to_period("Q").dt.to_timestamp()

            q_rows = []
            for q_start, grp in df_q.groupby("quarter_start"):
                q_rows.append({
                    "hs_code":       hs_code,
                    "quarter":       q_start,
                    "year":          str(q_start.year),
                    "quarter_label": f"{q_start.year}Q{q_start.quarter}",
                    "exp_dlr":       grp["exp_dlr"].sum(min_count=1),
                    "forecast":      grp["forecast"].sum(min_count=1),
                    "is_forecast":   int(grp["is_forecast"].max()),
                    "model_order":   df_m["model_order"].iloc[0],
                    "created_at":    created_at,
                })
            df_quarter = pd.DataFrame(q_rows)

            monthly_results.append(df_m)
            quarter_results.append(df_quarter)

        except Exception as e:
            failed_codes.append({"hs_code": hs_code, "error": str(e)})
            continue

    pbar.close()
    print("=" * 80)

    print(f"\n처리 완료:")
    print(f"  성공: {len(monthly_results):,}개")
    print(f"  실패: {len(failed_codes):,}개")

    if failed_codes:
        print(f"\n실패한 HS 코드 샘플 (최대 10개):")
        for item in failed_codes[:10]:
            print(f"  - {item['hs_code']}: {item['error']}")

    all_monthly = pd.concat(monthly_results, ignore_index=True) if monthly_results else pd.DataFrame()
    all_quarter = pd.concat(quarter_results, ignore_index=True) if quarter_results else pd.DataFrame()

    print(f"\n월별 데이터: {len(all_monthly):,} rows")
    print(f"분기별 데이터: {len(all_quarter):,} rows")
    print("\n모든 처리 완료!")

    return created_at, all_monthly, all_quarter


print("메인 함수 정의 완료")

메인 함수 정의 완료


## 2. DB 연결 정보 설정

In [13]:
from DATA.stock_invest_function import *

# DB 정보 입력
db_info = {
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'host': get_db_host(),
    'port': '3307',
    'database': 'investar'
}

# 연결 테스트
try:
    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
        f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
    )
    with engine.connect() as conn:
        print("DB 연결 성공")
    engine.dispose()
except Exception as e:
    print(f"DB 연결 실패: {e}")

DB 연결 성공


## 3. 데이터 로드

In [14]:
# 옵션 1: CSV 파일에서 로드
# trade_df = pd.read_csv("us_trade_data.csv")
# ── DB 연결 ───────────────────────────────────────────────
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
    f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

# ── 쿼리 (실제 컬럼명 기준) ───────────────────────────────
query = """
SELECT
    hs_code,
    date,
    exp_dlr            -- DB 실제 컬럼명 (expDlr 아님)
FROM us_export_data
WHERE exp_dlr IS NOT NULL
ORDER BY hs_code, date
"""

trade_df = pd.read_sql(query, engine)
engine.dispose()

# ── 후처리 ────────────────────────────────────────────────
trade_df['date'] = pd.to_datetime(trade_df['date'])

print(f"데이터 로드 완료: {len(trade_df):,} rows")
print(f"HS 코드 수: {trade_df['hs_code'].nunique():,}")
print(f"기간: {trade_df['date'].min()} ~ {trade_df['date'].max()}")

데이터 로드 완료: 59,094 rows
HS 코드 수: 500
기간: 2016-01-31 00:00:00 ~ 2026-03-31 00:00:00


## 4. 데이터 확인

In [15]:
# 기본 정보
print("데이터 구조:")
print(trade_df.head())

print("\n기초 통계:")
print(trade_df['exp_dlr'].describe())

# HS 코드별 관측치 수
hs_counts = trade_df.groupby('hs_code').size().sort_values(ascending=False)
print("\nHS 코드별 관측치 수 (상위 10개):")
print(hs_counts.head(10))

print(f"\n최소 관측치: {hs_counts.min()}")
print(f"최대 관측치: {hs_counts.max()}")
print(f"평균 관측치: {hs_counts.mean():.1f}")

데이터 구조:
  hs_code       date    exp_dlr
0  020130 2016-01-31  154815020
1  020130 2016-02-29  161201010
2  020130 2016-03-31  190582390
3  020130 2016-04-30  199209993
4  020130 2016-05-31  216155091

기초 통계:
count    5.909400e+04
mean     2.448497e+08
std      7.074420e+08
min      0.000000e+00
25%      5.886354e+07
50%      9.294453e+07
75%      1.740080e+08
max      1.492769e+10
Name: exp_dlr, dtype: float64

HS 코드별 관측치 수 (상위 10개):
hs_code
020130    123
847790    123
848340    123
848330    123
848310    123
848220    123
848210    123
848190    123
848180    123
848140    123
dtype: int64

최소 관측치: 50
최대 관측치: 123
평균 관측치: 118.2


In [16]:
# 테스트용 HS 코드 선택
test_hs = trade_df.groupby('hs_code').size().idxmax()
print(f"테스트 HS 코드: {test_hs}")
test_hs = '854232'
test_data = trade_df[trade_df['hs_code'] == test_hs].copy()
print(f"테스트 데이터: {len(test_data)} 관측치")
print(f"기간: {test_data['date'].min()} ~ {test_data['date'].max()}")

테스트 HS 코드: 020130
테스트 데이터: 123 관측치
기간: 2016-01-31 00:00:00 ~ 2026-03-31 00:00:00


## 6. 전체 HS 코드 예측 및 DB 저장

In [17]:
# 예측 파라미터 설정
FORECAST_MONTHS = 18  # 예측할 개월 수
MIN_OBS = 24          # 최소 관측치 수
IC = "aic"            # 정보 기준

print(f"예측 설정:")
print(f"  예측 개월: {FORECAST_MONTHS}")
print(f"  최소 관측치: {MIN_OBS}")
print(f"  정보 기준: {IC}")
print("\n실행하시겠습니까? (아래 셀 실행)")

예측 설정:
  예측 개월: 18
  최소 관측치: 24
  정보 기준: aic

실행하시겠습니까? (아래 셀 실행)


In [18]:
TARGET_HS = "854232"

hs_data = trade_df[trade_df["hs_code"] == TARGET_HS].copy()
hs_data


,hs_code,date,exp_dlr
45021,854232,2016-01-31,362971388
45022,854232,2016-02-29,279860085
45023,854232,2016-03-31,344666330
45024,854232,2016-04-30,309510794
45025,854232,2016-05-31,294998740
...,...,...,...
45139,854232,2025-11-30,211830171
45140,854232,2025-12-31,221421044
45141,854232,2026-01-31,285814153
45142,854232,2026-02-28,323864668


In [19]:
# 실행
created_at = forecast_and_save_all_hs_codes(
    trade_data=hs_data,
    db_info=db_info,
    forecast_months=FORECAST_MONTHS,
    min_obs=MIN_OBS,
    ic=IC,
    date_col="date"  # 또는 "date"
)

print(f"\n저장 완료 시각: {created_at}")

총 1개 HS 코드 처리 시작...
예측 개월 수: 18
생성 시각: 2026-05-14 18:44:46.095755


예측 진행: 100%|██████████| 1/1 [00:18<00:00, 18.79s/HS코드, 현재=854232, 성공=0, 실패=0]


처리 완료:
  성공: 1개
  실패: 0개

월별 데이터: 141 rows
분기별 데이터: 47 rows

모든 처리 완료!

저장 완료 시각: (datetime.datetime(2026, 5, 14, 18, 44, 46, 95755),     hs_code       date  year month      exp_dlr      forecast  is_forecast  \
0    854232 2016-01-31  2016    01  362971388.0  3.629714e+08            0   
1    854232 2016-02-29  2016    02  279860085.0  2.798601e+08            0   
2    854232 2016-03-31  2016    03  344666330.0  3.446663e+08            0   
3    854232 2016-04-30  2016    04  309510794.0  3.095108e+08            0   
4    854232 2016-05-31  2016    05  294998740.0  2.949987e+08            0   
..      ...        ...   ...   ...          ...           ...          ...   
136  854232 2027-05-31  2027    05          NaN  3.587178e+08            1   
137  854232 2027-06-30  2027    06          NaN  3.587177e+08            1   
138  854232 2027-07-31  2027    07          NaN  3.587177e+08            1   
139  854232 2027-08-31  2027    08          NaN  3.587177e+08            1   
140  85

## 7. DB 저장 결과 확인

In [20]:
# ════════════════════════════════════════════════════════════════════════════
# save_forecast_to_db.py
# 예측 결과 DataFrame → 기존 MySQL 테이블에 저장
#
# 기존 테이블 컬럼:
#   월별  : hs_code / date_month_end / expDlr / expDlr_forecast / is_forecast / params / created_at
#   분기별: hs_code / date_quarter_end / expDlr / expDlr_forecast / is_forecast / params / created_at
# ════════════════════════════════════════════════════════════════════════════

INSERT_MONTHLY = """
INSERT IGNORE INTO `us_trade_export_monthly_with_forecast`
    (hs_code, date_month_end, expDlr, expDlr_forecast,
     is_forecast, params, created_at)
VALUES
    (:hs_code, :date_month_end, :expDlr, :expDlr_forecast,
     :is_forecast, :params, :created_at)
"""

INSERT_QUARTER = """
INSERT IGNORE INTO `us_trade_export_quarter_with_forecast`
    (hs_code, date_quarter_end, expDlr, expDlr_forecast,
     is_forecast, params, created_at)
VALUES
    (:hs_code, :date_quarter_end, :expDlr, :expDlr_forecast,
     :is_forecast, :params, :created_at)
"""


def _clean_value(v):
    """
    MySQL 에 넣을 수 없는 값을 None 으로 치환
      - NaN, None, pd.NaT  → None
      - inf, -inf          → None  (pymysql ProgrammingError 방지)
      - pandas Timestamp   → python datetime  (타입 호환)
    """
    import math
    import datetime as dt

    if v is None:
        return None
    # pandas Timestamp → python datetime
    if isinstance(v, pd.Timestamp):
        return v.to_pydatetime()
    # float 계열 이상치
    if isinstance(v, float):
        if math.isnan(v) or math.isinf(v):
            return None
    # numpy scalar
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    return v


def save_forecast_to_db(
    all_monthly: pd.DataFrame,
    all_quarter: pd.DataFrame,
    db_info: Dict,
    chunk_size: int = 1000,
):
    """
    예측 결과 DataFrame 을 기존 MySQL 테이블에 저장합니다.
    UNIQUE KEY (hs_code, date) 기준 INSERT IGNORE → 중복 행 자동 스킵

    Parameters
    ----------
    all_monthly : forecast_and_save_all_hs_codes 의 두 번째 반환값
    all_quarter : forecast_and_save_all_hs_codes 의 세 번째 반환값
    db_info     : DB 접속 정보 dict
    chunk_size  : 한 번에 INSERT 할 행 수
    """
    from sqlalchemy import create_engine, text

    if all_monthly.empty and all_quarter.empty:
        print("[DB] 저장할 데이터가 없습니다.")
        return

    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
        f"@{db_info['host']}:{db_info['port']}/{db_info['database']}"
        f"?charset=utf8mb4"
    )

    with engine.connect() as conn:

        # ── 월별 저장 ─────────────────────────────────────────────────────
        if not all_monthly.empty:
            df_m = all_monthly.rename(columns={
                "date":        "date_month_end",
                "exp_dlr":     "expDlr",
                "forecast":    "expDlr_forecast",
                "model_order": "params",
            })[["hs_code", "date_month_end", "expDlr", "expDlr_forecast",
                "is_forecast", "params", "created_at"]].copy()

            df_m["date_month_end"] = pd.to_datetime(df_m["date_month_end"]).dt.date

            # inf / NaN / Timestamp → MySQL 안전값으로 변환
            rows_m = [
                {k: _clean_value(v) for k, v in row.items()}
                for row in df_m.to_dict(orient="records")
            ]

            ins_m = 0
            for i in range(0, len(rows_m), chunk_size):
                result = conn.execute(text(INSERT_MONTHLY), rows_m[i: i + chunk_size])
                ins_m += result.rowcount
            conn.commit()

            print(f"[월별]  신규 저장: {ins_m:,} 건 | 중복 스킵: {len(rows_m) - ins_m:,} 건")

        # ── 분기별 저장 ───────────────────────────────────────────────────
        if not all_quarter.empty:
            df_q = all_quarter.rename(columns={
                "quarter":     "date_quarter_end",
                "exp_dlr":     "expDlr",
                "forecast":    "expDlr_forecast",
                "model_order": "params",
            })[["hs_code", "date_quarter_end", "expDlr", "expDlr_forecast",
                "is_forecast", "params", "created_at"]].copy()

            df_q["date_quarter_end"] = pd.to_datetime(df_q["date_quarter_end"]).dt.date

            rows_q = [
                {k: _clean_value(v) for k, v in row.items()}
                for row in df_q.to_dict(orient="records")
            ]

            ins_q = 0
            for i in range(0, len(rows_q), chunk_size):
                result = conn.execute(text(INSERT_QUARTER), rows_q[i: i + chunk_size])
                ins_q += result.rowcount
            conn.commit()

            print(f"[분기별] 신규 저장: {ins_q:,} 건 | 중복 스킵: {len(rows_q) - ins_q:,} 건")

    engine.dispose()
    print("[DB] 저장 완료")


print("save_forecast_to_db 함수 정의 완료")

save_forecast_to_db 함수 정의 완료


In [21]:
# 예측 실행
created_at, all_monthly, all_quarter = forecast_and_save_all_hs_codes(
    trade_data=trade_df, db_info=db_info,
    forecast_months=FORECAST_MONTHS, min_obs=MIN_OBS,
    ic=IC, date_col="date"
)

# DB 저장
save_forecast_to_db(
    all_monthly=all_monthly,
    all_quarter=all_quarter,
    db_info=db_info,
)

총 500개 HS 코드 처리 시작...
예측 개월 수: 18
생성 시각: 2026-05-14 18:45:23.903028


예측 진행: 100%|██████████| 500/500 [2:36:18<00:00, 18.76s/HS코드, 현재=988000, 성공=499, 실패=0]  



처리 완료:
  성공: 500개
  실패: 0개

월별 데이터: 68,094 rows
분기별 데이터: 22,701 rows

모든 처리 완료!
[월별]  신규 저장: 68,094 건 | 중복 스킵: 0 건
[분기별] 신규 저장: 22,701 건 | 중복 스킵: 0 건
[DB] 저장 완료


## 8. 요약 통계

In [23]:
# 전체 예측 통계
engine = create_engine(
    f"mysql+pymysql://{db_info['user']}:{db_info['password']}@"
    f"{db_info['host']}:{db_info['port']}/{db_info['database']}"
)

query_summary = f"""
SELECT
    COUNT(DISTINCT hs_code) as hs_code_count,
    COUNT(*) as total_rows,
    SUM(is_forecast) as forecast_rows,
    MIN(date_month_end) as min_date,
    MAX(date_month_end) as max_date
FROM us_trade_export_monthly_with_forecast
WHERE created_at = '{created_at}'
"""

summary = pd.read_sql(query_summary, engine)
print("예측 요약 통계:")
print(summary)

engine.dispose()

예측 요약 통계:
   hs_code_count  total_rows forecast_rows min_date max_date
0              0           0          None     None     None


In [24]:
# ════════════════════════════════════════════════════════════════════════════
# check_forecast_db.py
# 특정 HS 코드의 DB 저장 상태 확인
# ════════════════════════════════════════════════════════════════════════════

def check_forecast_db(hs_code: str, db_info: Dict):
    """
    특정 HS 코드의 월별/분기별 예측 저장 상태를 확인합니다.

    Parameters
    ----------
    hs_code  : 확인할 HS 코드 (예: '854232')
    db_info  : DB 접속 정보 dict
    """
    from sqlalchemy import create_engine

    engine = create_engine(
        f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
        f"@{db_info['host']}:{db_info['port']}/{db_info['database']}"
        f"?charset=utf8mb4"
    )

    # ── 1. 월별 데이터 로드 ───────────────────────────────────────────────
    df_m = pd.read_sql(f"""
        SELECT *
        FROM us_trade_export_monthly_with_forecast
        WHERE hs_code = '{hs_code}'
        ORDER BY date_month_end
    """, engine)

    # ── 2. 분기별 데이터 로드 ─────────────────────────────────────────────
    df_q = pd.read_sql(f"""
        SELECT *
        FROM us_trade_export_quarter_with_forecast
        WHERE hs_code = '{hs_code}'
        ORDER BY date_quarter_end
    """, engine)

    engine.dispose()

    # ── 3. 기본 현황 출력 ─────────────────────────────────────────────────
    sep = "=" * 65
    print(sep)
    print(f"  HS 코드: {hs_code}")
    print(sep)

    # 월별 요약
    print("\n[월별 테이블] us_trade_export_monthly_with_forecast")
    if df_m.empty:
        print("  ※ 저장된 데이터 없음")
    else:
        actual   = df_m[df_m["is_forecast"] == 0]
        forecast = df_m[df_m["is_forecast"] == 1]
        print(f"  전체 행 수    : {len(df_m):,} 행")
        print(f"  실적 (is_forecast=0) : {len(actual):,} 행  "
              f"{actual['date_month_end'].min()} ~ {actual['date_month_end'].max()}")
        print(f"  예측 (is_forecast=1) : {len(forecast):,} 행  "
              f"{forecast['date_month_end'].min()} ~ {forecast['date_month_end'].max()}")
        print(f"  모델          : {df_m['params'].iloc[0]}")
        print(f"  created_at    : {df_m['created_at'].iloc[0]}")
        print(f"\n  --- 실적 마지막 5행 ---")
        print(actual[["date_month_end", "expDlr", "expDlr_forecast", "is_forecast"]].tail(5).to_string(index=False))
        print(f"\n  --- 예측 첫 5행 ---")
        print(forecast[["date_month_end", "expDlr", "expDlr_forecast", "is_forecast"]].head(5).to_string(index=False))

    # 분기별 요약
    print("\n[분기별 테이블] us_trade_export_quarter_with_forecast")
    if df_q.empty:
        print("  ※ 저장된 데이터 없음")
    else:
        actual_q   = df_q[df_q["is_forecast"] == 0]
        forecast_q = df_q[df_q["is_forecast"] == 1]
        print(f"  전체 행 수    : {len(df_q):,} 행")
        print(f"  실적 (is_forecast=0) : {len(actual_q):,} 행  "
              f"{actual_q['date_quarter_end'].min()} ~ {actual_q['date_quarter_end'].max()}")
        print(f"  예측 (is_forecast=1) : {len(forecast_q):,} 행  "
              f"{forecast_q['date_quarter_end'].min()} ~ {forecast_q['date_quarter_end'].max()}")
        print(f"\n  --- 실적 마지막 3행 ---")
        print(actual_q[["date_quarter_end", "expDlr", "expDlr_forecast", "is_forecast"]].tail(3).to_string(index=False))
        print(f"\n  --- 예측 첫 3행 ---")
        print(forecast_q[["date_quarter_end", "expDlr", "expDlr_forecast", "is_forecast"]].head(3).to_string(index=False))

    print(f"\n{sep}")
    return df_m, df_q


# ── 실행 ─────────────────────────────────────────────────────────────────
# HS 코드를 바꿔가며 확인
# df_monthly, df_quarter = check_forecast_db(
#     hs_code = "854232",   # ← 여기만 바꾸면 됩니다
#     db_info = db_info,
# )

In [25]:
# HS 코드만 바꿔서 실행
df_monthly, df_quarter = check_forecast_db(
    hs_code = "854232",
    db_info = db_info,
)

  HS 코드: 854232

[월별 테이블] us_trade_export_monthly_with_forecast
  전체 행 수    : 900 행
  실적 (is_forecast=0) : 796 행  2013-01-31 00:00:00 ~ 2026-03-31 00:00:00
  예측 (is_forecast=1) : 104 행  2025-10-31 00:00:00 ~ 2027-09-30 00:00:00
  모델          : {"model": "SARIMA", "order": [0, 1, 2], "seasonal_order": [0, 1, 1, 12], "aic": 4712.101536267766, "bic": 4723.414791216976}
  created_at    : 2025-12-13 23:00:31

  --- 실적 마지막 5행 ---
date_month_end      expDlr  expDlr_forecast  is_forecast
    2026-02-28 323864668.0      323864668.0            0
    2026-02-28 323864668.0      323864668.0            0
    2026-02-28 323864668.0      323864668.0            0
    2026-03-31 374446322.0      374446322.0            0
    2026-03-31 374446322.0      374446322.0            0

  --- 예측 첫 5행 ---
date_month_end  expDlr  expDlr_forecast  is_forecast
    2025-10-31     NaN     1.828625e+08            1
    2025-11-30     NaN     1.612057e+08            1
    2025-11-30     NaN     1.720582e+08            1

In [28]:
df_monthly

,hs_code,date_month_end,expDlr,expDlr_forecast,is_forecast,params,created_at
0,854232,2013-01-31,487586161.0,4.875862e+08,0,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2025-12-13 23:00:31
1,854232,2013-01-31,487586161.0,4.875862e+08,0,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2026-01-22 17:48:20
2,854232,2013-02-28,440591458.0,4.405915e+08,0,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2025-12-13 23:00:31
3,854232,2013-02-28,440591458.0,4.405915e+08,0,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2026-01-22 17:48:20
4,854232,2013-03-31,508646565.0,5.086466e+08,0,"{""model"": ""SARIMA"", ""order"": [0, 1, 2], ""seaso...",2025-12-13 23:00:31
...,...,...,...,...,...,...,...
895,854232,2027-08-31,NaN,2.825447e+08,1,"(0, 1, 1)(0, 0, 0, 12)",2026-04-13 23:12:26
896,854232,2027-08-31,NaN,3.587177e+08,1,"(1, 1, 0)(0, 0, 0, 12)",2026-05-14 08:51:07
897,854232,2027-08-31,NaN,3.587177e+08,1,"(1, 1, 0)(0, 0, 0, 12)",2026-05-14 18:45:23
898,854232,2027-09-30,NaN,3.587177e+08,1,"(1, 1, 0)(0, 0, 0, 12)",2026-05-14 08:51:07
